# Resonator Kappa through capacitance matrix simulation

In [1]:
%load_ext autoreload
%autoreload 2

## Rendering the design

In [2]:
import names as n
import design as d
from qdesignoptimizer.utils.chip_generation import create_chip_base

design, gui = create_chip_base(n.CHIP_NAME, d.chip_type, open_gui=True)
d.render_qiskit_metal_design(design, gui)

## Creating the study and optimization targets

In [3]:
import mini_studies as ms
import optimization_targets as ot

In [4]:
MINI_STUDY_GROUP = n.NBR_1
MINI_STUDY = ms.get_mini_study_res_feedline(group=MINI_STUDY_GROUP)
RENDER_QISKIT_METAL = lambda design, **kwargs: d.render_qiskit_metal_design(
    design, gui, **kwargs
)


opt_targets = ot.get_opt_target_res_kappa_feedline(group=MINI_STUDY_GROUP)

## Creating design analysis objects

In [5]:
import time
import parameter_targets as pt
import plot_settings as ps

from qdesignoptimizer.design_analysis import DesignAnalysis, DesignAnalysisState
from qdesignoptimizer.utils.utils import get_save_path

# Closing Ansys hfss
from qdesignoptimizer.utils.utils import close_ansys

close_ansys()

In [6]:
design_analysis_state = DesignAnalysisState(
    design, RENDER_QISKIT_METAL, pt.PARAM_TARGETS
)

design_analysis = DesignAnalysis(
    design_analysis_state,
    mini_study=MINI_STUDY,
    opt_targets=opt_targets,
    save_path=get_save_path("out/", n.CHIP_NAME),
    update_design_variables=False,
    plot_settings=ps.PLOT_SETTINGS_RESONATOR_KAPPA,
)

[INFO|2026-07-08 14:48:09]: self.eig_solver.sim.setup 
{
    "name": "Resonator_setup",
    "reuse_selected_design": true,
    "reuse_setup": true,
    "min_freq_ghz": 1,
    "n_modes": 1,
    "max_delta_f": 0.5,
    "max_passes": 10,
    "min_passes": 1,
    "min_converged": 1,
    "pct_refinement": 30,
    "basis_order": 1,
    "vars": {
        "Lj": "10 nH",
        "Cj": "0 fF"
    }
}
INFO 02:48PM [connect_project]: Connecting to Ansys Desktop API...
INFO 02:48PM [load_ansys_project]: 	Opened Ansys App
INFO 02:48PM [load_ansys_project]: 	Opened Ansys Desktop v2022.2.3
INFO 02:48PM [load_ansys_project]: 	Opened Ansys Project
	Folder:    C:/Users/nirpit/Documents/Ansoft/
	Project:   Project26
INFO 02:48PM [connect_design]: No active design found (or error getting active design).
INFO 02:48PM [connect]: 	 Connected to project "Project26". No design detected
02:48PM 19s WARNING [activate_ansys_design]: The design_name=get_mini_study_res_feedline was not in active project.  Designs in

## Optimization step

In [7]:
# number of runs of optimization and number of passes for simulation at each run
nbr_iterations = 10
group_passes_cap = 8

for i in range(nbr_iterations):
    design_analysis.update_nbr_passes_capacitance_ministudies(group_passes_cap)
    design_analysis.optimize_target({}, {})
    design_analysis.screenshot(gui=gui, run=i)

[INFO|2026-07-08 14:48:20]: Updated_design_vars
{
    "cpw_width": "10 um",
    "cpw_gap": "6 um",
    "design_var_width_qubit_1": "400um",
    "design_var_width_qubit_2": "400um",
    "design_var_cl_pos_x_qubit_1": "-100um",
    "design_var_cl_pos_y_qubit_1": "-300um",
    "design_var_cl_pos_x_qubit_2": "-100um",
    "design_var_cl_pos_y_qubit_2": "300um",
    "design_var_length_resonator_1_capacitance": "15um",
    "design_var_length_resonator_2_capacitance": "15um",
    "design_var_lj_qubit_1": "12.1nH",
    "design_var_lj_qubit_2": "9.1nH",
    "design_var_cj_qubit_1": "0fF",
    "design_var_cj_qubit_2": "0fF",
    "design_var_length_resonator_1": "7500um",
    "design_var_length_resonator_2": "6500um",
    "design_var_length_coupler_1to2": "4000um",
    "design_var_coupl_length_qubit_1_resonator_1": "100um",
    "design_var_coupl_length_qubit_2_resonator_2": "100um",
    "design_var_coupl_length_resonator_1_tee": "400um",
    "design_var_coupl_length_resonator_2_tee": "160um"
}
[I

## Results

### Cross Kerr

In [ ]:
design_analysis.get_cross_kerr_matrix(iteration=-1)

## Update parameters

In [ ]:
design_analysis.overwrite_parameters()

## Close

In [ ]:
from qdesignoptimizer.utils.utils import close_ansys

close_ansys()

## Example result
![image info](plots/kappa_target_resonator.png)